# H03：实际成交、假设接收的分钟特征验证

这是技术一致性实验。价格和成交量来自本地正式逐笔对象；订阅、断线和消息接收时间为预先冻结的人工情景。成功只支持这些情景下的计算一致性，不能证明真实接收行为或 alpha。

方案、验收和最终结论的唯一入口是 [README 的 H03](README.md#h03-已收逐笔的分批与离线特征一致性)。H02 的失败结果不变。

**执行结果：90 次分批/离线对照全部相等；14,154 笔实际成交；两个错误方向处理的负对照均被识别。真实实盘一致性和 alpha 未验证。**

## 1. 固定输入与执行

两只股票、同一日期，三种情景、五个截止时点，分别使用 1、17、257 行消息批次。底层分钟聚合复用仓库正式 builder；研究层负责接收裁剪、连续区间重置和方向重建。

In [1]:
from pathlib import Path
import json
import runpy
import pandas as pd

source = Path('/home/wsw/app/dev/trading')
driver = runpy.run_path(str(source / 'research/subscription-training/validate_reception.py'))
evidence = driver['run_reception_validation'](
    source_root=source,
    storage_root=Path('/home/wsw/app/data'),
    evidence_parent=Path('/home/wsw/app/research-evidence'),
)
print(str(evidence))

/home/wsw/app/research-evidence/subscription-reception-2026-09-19-nqemd547


## 2. 可恢复输入

归档实际使用的逐笔子集、来源 Meta、源码及 lock；不以文件修改时间充当历史可用时间。

In [2]:
run = json.loads((evidence / 'run.json').read_text())
pd.DataFrame(run['origins'])[['symbol', 'retained_rows', 'retained_table_sha256']]

,symbol,retained_rows,retained_table_sha256
0,600000,6210,005fffe9672a389ddc149625b88689edcf676e579a251d...
1,000001,7944,6352183459aedacbbd86d8d1b5820951ee5016f7af8142...


## 3. 分批与离线结果

比较表必须精确相等。成交笔数、成交量另行精确核对；金额用独立求和核对，仅容纳冻结的浮点加法误差。

In [3]:
checks = pd.DataFrame(json.loads((evidence / 'checks.json').read_text()))
checks.groupby(['scenario', 'batch_size'], sort=True).agg(
    snapshots=('batch_equals_offline', 'size'),
    all_equal=('batch_equals_offline', 'all'),
)

snapshots  all_equal
scenario   batch_size                      
disconnect 1                  10       True
           17                 10       True
           257                10       True
regular    1                  10       True
           17                 10       True
           257                10       True
reordered  1                  10       True
           17                 10       True
           257                10       True

## 4. 接收区间和冷启动

连续连接覆盖与成交是否存在分别记录。14:12..14:13 的断线不能由已存在的历史分钟制品补回，14:14 不能声明已连续观察五分钟。

In [4]:
coverage = checks.loc[(checks['scenario'] == 'disconnect') & (checks['batch_size'] == 17)].copy()
coverage['decision_shanghai'] = pd.to_datetime(coverage['decision_ts_utc'], unit='us', utc=True).dt.tz_convert('Asia/Shanghai').dt.strftime('%H:%M')
coverage[['symbol', 'decision_shanghai', 'received_prefix_rows', 'window_trade_count', 'connection_covers_window']]

,symbol,decision_shanghai,received_prefix_rows,window_trade_count,connection_covers_window
65,600000,14:04,344,344,False
66,600000,14:10,1419,800,True
67,600000,14:14,245,245,False
68,600000,14:20,1224,844,True
69,600000,14:30,4279,1278,True
80,000001,14:04,0,0,False
81,000001,14:10,1067,1067,True
82,000001,14:14,1856,891,True
83,000001,14:20,2643,656,True
84,000001,14:30,6261,1942,True


## 5. 负对照和结论边界

确定性小反例分别检测继承全日方向、按到达顺序计算方向这两种错误做法；此外污染未接收价格及全部既存方向，核对合法快照不变。这里不建立供应商消息完整性或真实订单簿重建结论。

In [5]:
summary = json.loads((evidence / 'summary.json').read_text())
summary

{'comparisons': 90,
 'all_comparisons_equal': True,
 'independent_count_volume_notional_checks': True,
 'unreceived_and_persisted_direction_poison_checks': True,
 'past_snapshots_unchanged': True,
 'negative_controls': {'inherited_direction_rejected': True,
  'arrival_order_direction_rejected': True},
 'real_time_alignment_verified': False,
 'alpha_verified': False}